# U4-04 · Seudonimización y k-anonimato

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 4, apartado 4.9*

Ejecuta cada celda con **Mayús + Intro**. Todos los datos son ficticios y se crean dentro del propio notebook.

Los DNI de este notebook son ficticios.

## 1. Seudonimizar con HMAC

In [ ]:
import hmac, hashlib
import pandas as pd

clave = b"clave-secreta-de-ejemplo"     # en un caso real, guardada fuera del código

def seudonimo(valor):
    return hmac.new(clave, valor.encode(), hashlib.sha256).hexdigest()[:12]

ops = pd.DataFrame({"operario": ["12345678Z", "87654321X", "12345678Z"],
                    "piezas":   [410, 385, 402]})
ops["operario"] = ops["operario"].map(seudonimo)
ops

In [ ]:
# El mismo operario conserva el mismo seudónimo: se puede seguir agregando
ops.groupby("operario")["piezas"].sum()

In [ ]:
# Con otra clave, los seudónimos cambian por completo
clave = b"otra-clave"
seudonimo("12345678Z")

## 2. ¿Cumple un conjunto el k-anonimato?

Un conjunto es *k*-anónimo si cada combinación de atributos que podrían identificar a alguien (cuasi-identificadores) aparece al menos *k* veces (Sweeney, 2002).

In [ ]:
plantilla = pd.DataFrame({
    "edad":          [34, 35, 52, 29, 41, 38, 33, 57, 45, 31],
    "codigo_postal": ["38001", "38001", "38002", "38001", "38003", "38002", "38001", "38002", "38003", "38001"],
    "sexo":          ["M", "M", "H", "H", "M", "H", "M", "H", "M", "H"],
    "bajas_dias":    [3, 0, 12, 1, 5, 0, 2, 20, 4, 0],
})
cuasi = ["edad", "codigo_postal", "sexo"]
print("k =", plantilla.groupby(cuasi).size().min())   # con la edad exacta, cada persona es única

In [ ]:
# Generalizar: edad en tramos de 10 años y código postal recortado
g = plantilla.copy()
g["edad"] = (g["edad"] // 10 * 10).astype(str) + "-" + (g["edad"] // 10 * 10 + 9).astype(str)
g["codigo_postal"] = g["codigo_postal"].str[:4] + "*"
print("k =", g.groupby(cuasi).size().min())
g.groupby(cuasi).size()

**Ejercicio.** ¿Qué generalización adicional haría falta para conseguir *k* = 2? ¿Qué información se pierde al hacerlo?

> Sweeney, L. (2002). k-Anonymity: A Model for Protecting Privacy. *IJUFKS*, 10(5), 557–570. https://doi.org/10.1142/S0218488502001648